# Session 1 — Welcome to Wavely: Data Engineering & Python basics

You just joined **Wavely**, a (fictional) music streaming app popular in Central Europe.
Every time someone presses ▶, the app sends a small message to our servers. Our job as data engineers:
turn millions of these messages into clean, trustworthy data that the rest of the company can use.

**Today you will learn**
1. What a data engineer does (and how it differs from analysts and data scientists)
2. What this course covers, session by session, and what you will build
3. How to use a notebook
4. Python basics: values, variables, strings, lists and dictionaries

**How these notebooks work**
- **Core** exercises: everyone does these.
- **Stretch** exercises: for when you finish early or already know some Python.
- **Under the hood** boxes: extra background. Nice to know, never required.

---
## Part 1 — What is data engineering?

Follow one song play through Wavely:

```
 phone app ──► event "U0042 played T0128 for 183 s" ──► raw storage ──► pipeline (clean, combine) ──► tables
                                                                                                   │
                     dashboards · recommendations · royalty payments to artists · finance  ◄───────┘
```

| Role | Main question | Typical tools |
|---|---|---|
| **Data engineer** | *How do we get reliable data from A to B, every day, automatically?* | Python, Spark, SQL, cloud storage, orchestration |
| **Data analyst** | *What happened, and why?* | SQL, dashboards, spreadsheets |
| **Data scientist** | *What will happen? What should we recommend?* | Python, statistics, machine learning |

Without data engineers, analysts and scientists spend most of their time fixing messy data instead of answering questions.

**Discuss with your neighbour (2 min):** Wavely pays artists per play. What could go wrong if the same play is counted twice? If a play is lost?

### The data engineering lifecycle

Whatever company you work for, data engineering work falls into these stages. This course touches every one of them:

| Stage | Question | In this course |
|---|---|---|
| **Sourcing / ingestion** | How do we get the data in? | files, SFTP, S3, databases, APIs (sessions 2, 6, 7) |
| **Storage** | Where and in which format do we keep it? | CSV vs Parquet, partitions, data lake layers (sessions 4, 5) |
| **Transformation** | How do we clean, combine and aggregate it? | Python, then Spark (sessions 2-5) |
| **Quality** | Can we trust it? | quality checks, quarantine, reconciliation (session 5) |
| **Orchestration** | How does it run every day without us? | incremental loads, scheduling, Airflow/Dagster (session 8) |
| **Serving** | Who uses it, and how? | gold tables, charts, the business questions (sessions 5, 9) |

Under all of it: **security** (credentials, least privilege), **version control** (Git) and **cost** (don't re-process what you don't need to).

---
## The course at a glance

| # | Session | What you will learn | What you will build |
|---|---|---|---|
| 1 | **Welcome + Python I** | the data engineer's job; values, strings, lists, dicts | describe plays and playlists as Python data |
| 2 | **Python II** | `if`, loops, functions, errors, reading/writing files | clean a messy listening log in plain Python |
| 3 | **Meet Spark** | how Spark spreads work over many machines; DataFrames, schemas, lazy evaluation | explore Wavely's catalogue and find data problems |
| 4 | **Spark: clean, join, save** | cleaning functions, `groupBy`, joins, CSV vs Parquet | answer: top artists, skip rate per genre, free vs premium |
| 5 | **Building a pipeline** | ETL vs ELT, bronze/silver/gold, quality checks, idempotency | Wavely's first real pipeline, from raw files to charts |
| 6 | **Sourcing data I** | pull vs push, credentials, **SFTP**, **S3 buckets**, databases, change data capture | download partner files from SFTP, weather data from a real AWS bucket, incremental database pulls |
| 7 | **Sourcing data II** | JSON, **APIs**, nested data, **Git** | enrich our artists with live data from the Deezer API |
| 8 | **Incremental & orchestration** | process only new data, scheduling, Airflow/Dagster, data lakes, warehouses, lakehouses | a pipeline that runs every "night" without double counting |
| 9 | **Capstone** | presenting a data product | your own pipeline, presented to the class |

**Tools:** Python · PySpark (the same Spark you will use on clusters in the distributed processing course) · Google Colab · Git/GitHub.
SQL is covered in your separate SQL course, so here we transform data with Python and Spark's DataFrame API.

**Capstone (sessions 6-9):** you build your own bronze → silver → gold pipeline on one of three datasets — podcasts (getting started),
concert tickets (medium) or playlists (advanced) — with quality checks, and present it in session 9.

**Homework:** after every session (about 45-60 minutes). We always solve it together at the start of the next session, so it's fine to get stuck.

**By the end you can** write Python to work with data, process data with Spark, get data from files, servers, cloud storage, databases and APIs,
build a layered pipeline that checks its own quality and can be re-run safely, and explain how companies schedule and run these pipelines in the cloud.

---
## Part 2 — Notebooks in 60 seconds

A notebook is a list of **cells**:
- **Text cells** (like this one) explain things.
- **Code cells** contain Python. Click into one and press **Shift + Enter** to run it. The output appears below.

Run the cell below.

In [ ]:
print("Hello, Wavely!")

Things to know:
- Cells run in the order **you** run them, not the order on the page. If things get weird: *Runtime → Restart and run all* (Colab) or *Kernel → Restart* (Jupyter).
- The last line of a cell is displayed automatically, so `print` is not always needed.
- Text after `#` is a **comment**: Python ignores it. We use comments to explain code.

In [ ]:
# This is a comment. The next line is a calculation; its result is shown because it is the last line.
3 + 4

---
## Part 3 — Values and variables

Python works with **values** of different **types**:

| Type | Example | Used for |
|---|---|---|
| `int` | `183` | whole numbers (counts, milliseconds) |
| `float` | `3.05` | decimals (prices, averages) |
| `str` | `"Dua Lipa"` | text — always in quotes |
| `bool` | `True` / `False` | yes/no facts (was the song skipped?) |
| `None` | `None` | "no value" / missing data |

A **variable** is a name for a value: `name = value`.

In [2]:
artist = "Dua Lipa"
ms_played = 183000
skipped = False
country = None  # the app did not send a country

print(artist, ms_played, skipped, country)
print(type(artist), type(ms_played), type(skipped), type(country))

Dua Lipa 183000 False None
<class 'str'> <class 'int'> <class 'bool'> <class 'NoneType'>


Arithmetic works as you'd expect. Two operators are very handy for durations:
- `//` integer division (how many *whole* times does it fit?)
- `%` remainder (what is left over?)

In [3]:
seconds = ms_played / 1000  # / always gives a float
print(seconds)

minutes = 183 // 60  # whole minutes
rest = 183 % 60  # remaining seconds
print(minutes, rest)

183.0
3 3


### Exercise 1 (Core) — Milliseconds to `m:ss`
The app reports durations in milliseconds, but the player shows `m:ss`.
Given `ms = 225000`, compute `minutes` and `seconds` (both `int`).

In [ ]:
ms = 225000


# YOUR CODE HERE

print(minutes, seconds)  # expected: 3 45

---
## Part 4 — Strings

Strings have **methods**: functions you call with a dot, like `text.lower()`. They return a *new* string.

In [ ]:
raw_artist = "   dUA lIPA  "

print(raw_artist.strip())  # remove spaces at both ends
print(raw_artist.lower())
print(raw_artist.upper())
print(raw_artist.strip().title())  # methods can be chained
print(raw_artist.replace("U", "u"))

**f-strings** put values inside text. Put an `f` before the quote and variables in `{ }`:

In [ ]:
title = "Neon Rain"
plays = 1520
print(f"{title} by {artist} was played {plays} times")
print(f"Average: {183.456:.1f} seconds")  # :.1f = one decimal

**Indexing and slicing:** characters are numbered from 0. `text[0]` is the first, `text[-1]` the last,
`text[0:3]` the first three. **Splitting** turns one string into a list of parts.

In [4]:
event_id = "7a28c2bc7c9e"
print(event_id[0], event_id[-1], event_id[0:4])

line = "Dua Lipa - Neon Rain"
parts = line.split(" - ")
print(parts)
print(parts[0], "|", parts[1])

7 e 7a28
['Dua Lipa', 'Neon Rain']
Dua Lipa | Neon Rain


### Exercise 2 (Core) — Clean an artist name
Data from apps is messy. Turn `messy = "  the WEEKND "` into `"The Weeknd"` and store it in `clean`.

Then build the sentence `"Now playing: The Weeknd"` with an f-string and store it in `message`.

In [ ]:
messy = "  the WEEKND "

# YOUR CODE HERE

print(repr(clean))  # repr shows the quotes, so you can spot leftover spaces
print(message)

### Exercise 2b (Stretch) — Parse a duration string
Some devices send durations as text like `"3:45"`. Convert it to total seconds (`225`) using `split` and `int(...)`.

In [ ]:
duration_text = "3:45"

# YOUR CODE HERE

print(total)  # expected: 225

---
## Part 5 — Lists

A **list** holds many values in order: `[a, b, c]`. Lists are how we hold *columns* of data for now.

In [ ]:
minutes_per_day = [42, 15, 63, 0, 88, 120, 97]  # Monday .. Sunday

print(len(minutes_per_day))  # how many values
print(minutes_per_day[0])  # first (Monday)
print(minutes_per_day[-2:])  # last two (weekend)
print(sum(minutes_per_day), max(minutes_per_day), min(minutes_per_day))
print(sorted(minutes_per_day, reverse=True))
print(88 in minutes_per_day)

minutes_per_day.append(30)  # lists can grow
print(minutes_per_day)

### Exercise 3 (Core) — A week of listening
Using `week` below:
1. `total` — total minutes in the week
2. `average` — average minutes per day, rounded to 1 decimal (`round(x, 1)`)
3. `top3` — the three biggest values, largest first
4. `weekend_share` — share of minutes listened on Saturday + Sunday, in percent, rounded to 1 decimal

In [ ]:
week = [35, 20, 48, 51, 60, 140, 110]  # Monday .. Sunday

# YOUR CODE HERE

print(total, average, top3, weekend_share)  # expected: 464 66.3 [140, 110, 60] 53.9

---
## Part 6 — Dictionaries

A **dictionary** (`dict`) stores **key → value** pairs. It is the perfect shape for *one event* or *one row* of data.

In [ ]:
play = {
    "event_id": "7a28c2bc7c9e",
    "user_id": "U0525",
    "track_id": "T0028",
    "ms_played": 226000,
    "skipped": False,
}

print(play["user_id"])  # read a value by key
play["device"] = "desktop"  # add a new key
play["ms_played"] = 225000  # change a value
print(play)
print(list(play.keys()))

Asking for a key that does not exist crashes with a `KeyError`. Real data often has missing fields,
so use `.get(key, default)` instead:

In [ ]:
print(play.get("country"))  # None, no crash
print(play.get("country", "unknown"))  # with a default value

Values can be dictionaries themselves (**nesting**). This is exactly what data from apps and websites
looks like (it is called **JSON**, and we will meet it again in session 7).

In [ ]:
event = {
    "event_id": "f502f73a3acf",
    "event_type": "search",
    "user": {
        "id": "U0624",
        "plan": "premium",
        "device": {"type": "mobile", "os": "android"},
    },
    "payload": {"query": "queen", "results": 27},
}

print(event["user"]["plan"])
print(event["user"]["device"]["os"])

### Exercise 4 (Core) — Work with an event
Using `event` from above:
1. Store the search text in `query`.
2. Store the app version in `app_version`. It is **not** in the event, so use `.get` with default `"unknown"` on the device dict.
3. Store `True` in `is_premium` if the user's plan is `"premium"` (hint: `==` compares two values and gives a `bool`).

In [ ]:
# YOUR CODE HERE

print(query, app_version, is_premium)  # expected: queen unknown True

---
## Part 7 — A list of dictionaries is a table

Put many row-dicts in a list and you have a **table**. Every dict is a row, every key is a column.
This idea is the basis of everything we do with Spark from session 3 on.

In [ ]:
plays = [
    {"user_id": "U0525", "artist": "Dua Lipa", "seconds": 226, "skipped": False},
    {"user_id": "U0277", "artist": "Drake", "seconds": 6, "skipped": True},
    {"user_id": "U0458", "artist": "Coldplay", "seconds": 280, "skipped": False},
]

print(len(plays), "rows")
print(plays[1])  # second row
print(plays[1]["artist"])  # one cell

### Exercise 5 (Core) — Add a row and summarise
1. Append a new play: user `"U0554"`, artist `"Queen"`, `354` seconds, not skipped.
2. Compute `total_seconds`: the sum of `seconds` of all 4 rows (no loops yet — index each row).
3. Compute `skip_count`: how many of the 4 rows are skipped. Hint: `True` counts as `1` and `False` as `0` when added.

In [ ]:
# YOUR CODE HERE

print(len(plays), total_seconds, skip_count)  # expected: 4 866 1

Writing `plays[0]`, `plays[1]`, ... by hand does not scale to 4,000 rows — next session we learn **loops**.

### Exercise 5b (Stretch) — Sneak peek
If you already know loops or comprehensions: compute `total_seconds` again in one line that would work for any number of rows,
and build `artists`, a sorted list of the distinct artist names.

In [ ]:
# YOUR CODE HERE

print(total_seconds, artists)

> **Under the hood:** Python lists and dicts live in your computer's memory (RAM). A laptop has maybe 8-16 GB.
> Wavely-scale data (billions of plays) is far larger, which is why from session 3 we use **Spark**, which spreads data over many computers.

---
## Wrap-up

| Concept | Example |
|---|---|
| variable | `artist = "Dua Lipa"` |
| types | `int`, `float`, `str`, `bool`, `None` |
| string methods | `.strip()`, `.lower()`, `.title()`, `.split()`, f-strings |
| list | `[42, 15, 63]`, `len`, `sum`, `sorted`, `append` |
| dict | `{"user_id": "U0525"}`, `d["key"]`, `d.get("key", default)` |
| table | a list of dicts |

**Homework:** `homework/hw01_python_basics.ipynb` — we go through it together at the start of session 2.